# Programs and optimizers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/core/07_program.ipynb)

Every finite-dimensional optimization problem in minilink—controller tuning, co-design, trajectory optimization, and MPC—lowers to one canonical description, `MathematicalProgram`, solved by `Optimizer` into an `OptimizationResult`.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. `MathematicalProgram`, `Optimizer`, and `OptimizationResult`: public API

A `MathematicalProgram` describes the nonlinear program

$$\min_{z \in \mathbb{R}^{n_z}} J(z; p) \quad \text{s.t.} \quad h(z; p) = 0,\quad g(z; p) \ge 0,\quad \text{lower} \le z \le \text{upper}.$$

### `MathematicalProgram` constructor and attributes

| Field | Type & signature | Default | Meaning |
| --- | --- | --- | --- |
| `n_z` | `int` | required | Decision vector dimension ($z \in \mathbb{R}^{n_z}$) |
| `J` | `Callable[[z], float]` or `Callable[[z, params], float]` | required | Scalar objective function |
| `grad_J`, `hess_J` | `Callable \| None` | `None` | Optional explicit gradient `(n_z,)` and Hessian `(n_z, n_z)` (auto-differentiated on `"jax"` or finite-differenced on `"numpy"` when omitted) |
| `h`, `jac_h` | `Callable -> array (n_h,) \| None` | `None` | Equality constraints $h(z) = 0$ and optional Jacobian `(n_h, n_z)` |
| `g`, `jac_g` | `Callable -> array (n_g,) \| None` | `None` | Inequality margins $g(z) \ge 0$ and optional Jacobian `(n_g, n_z)` |
| `lower`, `upper` | `array_like (n_z,) \| None` | `[-inf, ...]`, `[+inf, ...]` | Box bounds on the decision vector $z$ |
| `params` | `dict \| None` | `None` | Default parameter tree for a parametric program |
| `backend` | `"numpy" \| "jax"` | `"numpy"` | Array backend of the program's callables |
| `metadata` | `dict` | `{}` | Auxiliary metadata (e.g. `metadata["z0"]` for exported programs) |

### `Optimizer` constructor and methods

| Method / Property | Arguments | Output type & shape | Meaning |
| --- | --- | --- | --- |
| `Optimizer(program, z0=None, ...)` | `program: MathematicalProgram`, `z0: array (n_z,) \| None = None`, `*, method="scipy_slsqp"`, `backend=None`, `use_hessian=False`, `options=None`, `iterations=None` | `Optimizer` | Compile `program` once and bind it to a solver preset |
| `opt.params` | — | `dict` | Program parameters plus `"method"` hyperparameters (e.g. `learning_rate` for `"gd"` / `"adam"`) |
| `opt.program_evaluator` | — | `ProgramEvaluator` | Compiled evaluator providing `.objective(z)`, `.gradient(z)`, `.constraint_violations(z)` |
| `opt.solve(...)` | `*, params=None, z0=None, callback=None, record_solve_time=False, verbose=False` | `OptimizationResult` | Solve the program from `z0` (or default `opt.z0`) at `params` without recompiling |
| `opt.as_step_system(params=None)` | `params: dict \| None = None` | `IterationSystem (StepSystem)` | View a native first-order optimizer (`"gd"`, `"adam"`) as a discrete-time `StepSystem` whose state is the iterate |

### `OptimizationResult` fields

| Field | Type & shape | Meaning |
| --- | --- | --- |
| `result.z` | `ndarray (n_z,)` (or traced JAX array) | Optimal decision vector $z^\star$ |
| `result.success` | `bool` | Solver convergence flag |
| `result.cost` | `float` | Objective value $J(z^\star)$ |
| `result.message` | `str` | Solver status message |
| `result.stats` | `dict[str, Any]` | Solver statistics (`{"nit": ...}`, function evaluations, etc.) |
| `result.solve_time_s` | `float \| None` | Wall-clock solve duration in seconds (populated when `record_solve_time=True` or `verbose=True`) |

## 2. Tuning a controller with `pe.program(wrt)`, `pe.pack`, and `pe.unpack`

`pe.program(wrt)` exports a `MathematicalProgram` whose decision vector $z$ packs the parameter paths listed in `wrt`, whose objective is `pe.evaluate(params).objective`, and whose inequality constraint $g(z) \ge 0$ is the reachability/feasibility margin of the problem's sets:

In [ ]:
import numpy as np

from minilink import (
    PID,
    BallSet,
    InvertedPendulum,
    Optimizer,
    PlanningProblem,
    QuadraticCost,
)

plant = InvertedPendulum()
cost = QuadraticCost.from_system(plant, Q=np.diag([10.0, 1.0]), R=0.1 * np.eye(1))
problem = PlanningProblem(
    plant,
    cost=cost,
    x_start=np.array([0.4, 0.0]),
    tf=4.0,
    Xf=BallSet(np.zeros(2), 0.05),
)
pe = problem.compile(PID(Kp=20.0, Kd=4.0) % 0.01)

wrt = ["controller.Kp", "controller.Kd"]
program = pe.program(wrt)
print(program)

In [ ]:
opt = Optimizer(program, method="auto")
best = opt.solve(record_solve_time=True)
print(best)

In [ ]:
best.z, best.success, best.cost, best.stats["nit"], best.solve_time_s is not None

In [ ]:
# Pack and unpack between the parameter tree and flat vector z
z_initial = pe.pack(pe.params, wrt)
tuned_params = pe.unpack(best.z, wrt)
z_initial, pe.pack(tuned_params, wrt), pe.evaluate(tuned_params)

In [ ]:
pe.loop.plot_trajectory(pe.trajectory(tuned_params))

## 3. Writing a constrained `MathematicalProgram` by hand

Solver presets accepted by `Optimizer(program, z0, method=...)`:

| `method` | Algorithm | Admits |
| --- | --- | --- |
| `"auto"` | `"l-bfgs"` when unconstrained or box-bounded, `"scipy_slsqp"` when constrained | Any program |
| `"l-bfgs"` | Quasi-Newton L-BFGS-B | Unconstrained or box-bounded |
| `"scipy_slsqp"`, `"scipy_trust_constr"` | Sequential quadratic programming / interior trust region | Equality, inequality, and box constraints |
| `"ipopt"` | Interior point (`cyipopt` optional dependency) | Equality, inequality, and box constraints |
| `"gd"`, `"adam"` | Projected gradient descent / Adam (`iterations=K` fixes step count for pure JAX scan) | Unconstrained or box-bounded |

In [ ]:
import jax.numpy as jnp

from minilink import MathematicalProgram


def J(z):
    return 0.5 * (z @ z)


def h(z):
    return jnp.array([z[0] + z[1] - 1.0])


def g(z):
    return jnp.array([z[0] - 0.7])


nlp = MathematicalProgram(
    n_z=2,
    J=J,
    h=h,
    g=g,
    lower=[0.0, 0.0],
    upper=[1.0, 1.0],
    backend="jax",
)
nlp_opt = Optimizer(nlp, z0=np.array([0.5, 0.5]), method="auto")
solution = nlp_opt.solve()
solution.z, solution.success, solution.cost

`nlp_opt.program_evaluator` exposes the compiled objective, gradient, equality residual, inequality margin, constraint Jacobians, and violation summary `(eq_inf, min_ineq, bound_inf)` at any vector $z$:

In [ ]:
ev = nlp_opt.program_evaluator
(
    ev.objective(solution.z),
    ev.gradient(solution.z),
    ev.equality_residual(solution.z),
    ev.inequality_margin(solution.z),
    ev.constraint_violations(solution.z),
)

## 4. Parametric programs and differentiable fixed-iteration solves

When `params=` is passed to `MathematicalProgram`, its callables take `(z, params)` and `optimizer.solve(params=...)` solves for new parameter values without recompiling.

With `method="gd"` or `"adam"` and `iterations=K`, `solve(params=..., z0=...)` executes a pure $K$-step `lax.scan` that is directly differentiable by `jax.grad` and `jax.jacobian`, and `as_step_system()` exposes the optimizer itself as a `StepSystem`:

In [ ]:
def J_target(z, params):
    e = z - params["z_bar"]
    return 0.5 * (e @ e)


target_program = MathematicalProgram(
    n_z=2,
    J=J_target,
    params={"z_bar": np.array([1.0, -0.5])},
    backend="jax",
)
lbfgs_opt = Optimizer(target_program, z0=np.zeros(2), method="l-bfgs")
lbfgs_opt.solve().z, lbfgs_opt.solve(params={"z_bar": np.array([3.0, 2.0])}).z

In [ ]:
descent = Optimizer(
    target_program,
    z0=np.zeros(2),
    method="gd",
    options={"learning_rate": 0.3},
    iterations=20,
)
iter_sys = descent.as_step_system()
iterates = iter_sys.compute_rollout(20)
iter_sys.n, iter_sys.p, iterates.x.shape

In [ ]:
iter_sys.plot_rollout(iterates)

In [ ]:
import jax

dz_dz_bar = jax.jacobian(lambda z_bar: descent.solve(params={"z_bar": z_bar}).z)(
    jnp.array([1.0, -0.5])
)
dz_dz_bar

Next, [`08_planners`](08_planners.ipynb): `Planner`, `PlanningSolution`, `compare`, `CompiledTrajectoryPlanner`, and `ModelPredictiveController`.